In [25]:
from pathlib import Path
from tqdm import tqdm
import sys
import re
import numpy as np
import torch

In [2]:
sys.path.append("..")

In [3]:
EXPERIMENT_DIR = Path("../out/memorize-replication/small-prototypes")

In [4]:
print(*[str(p.name) for p in EXPERIMENT_DIR.glob("*")], sep="\n")

2025-09-18T16-19-11Z__single-domain-80k-prototype__49346ed7__s1024__44e2826__3bb59cfc


In [60]:
experiment = "2025-09-18T19-51-43Z__single-domain-80k-prototype__49346ed7__s1024__7cbbf08__ac884235"

In [61]:
experiment_dir = EXPERIMENT_DIR / experiment
checkpoint_dir = experiment_dir / "checkpoints"

In [62]:
# Match the pattern step-\d+ to get a list of step numbers
step_numbers = sorted([
    int(match.group(1))
    for p in checkpoint_dir.glob("*")
    if (match := re.search(r"step-(\d+)", str(p.name))) is not None
])

In [63]:
selected_checkpoint_number = step_numbers[0]

In [64]:
step_dir = checkpoint_dir / f"step-{selected_checkpoint_number:06d}"

In [65]:
model_file = step_dir / "model.pt"
config_file = experiment_dir / "meta" / "config.toml"

In [66]:
from src.model import GPT
from src.config.manager import ConfigManager

config_manager = ConfigManager()
config_manager.load_from_toml_file(config_file)
config_manager.config.model


Model(n_layer=1, n_head=8, n_embd=32, block_size=64, dropout=0.0, bias=True, weight_tying=True, vocab_size=2048)

In [67]:
device = "cuda:7"

In [68]:
model = GPT(config_manager.config.model)
model.to(device)
model.eval()

number of parameters: 80.35K


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(2048, 32)
    (wpe): Embedding(64, 32)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0): Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=32, out_features=96, bias=True)
          (c_proj): Linear(in_features=32, out_features=32, bias=True)
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=32, out_features=128, bias=True)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=128, out_features=32, bias=True)
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=32, out_features=2048, bias=False)
)

In [69]:
data_dir = Path("../data/random_V2048_L64_N65536")

In [70]:
def get_batch(data_dir, block_size=64, batch_size=1):
    # We recreate np.memmap every batch to avoid a memory leak, as per
    # https://stackoverflow.com/questions/45132940/numpy-memmap-memory-usage-want-to-iterate-once/61472122#61472122
    data = np.memmap(data_dir / "train.bin", dtype=np.uint16, mode="r")
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack(
        [torch.from_numpy((data[i : i + block_size]).astype(np.int64)) for i in ix]
    )
    y = torch.stack(
        [
            torch.from_numpy((data[i + 1 : i + 1 + block_size]).astype(np.int64))
            for i in ix
        ]
    )
    # pin arrays x,y, which allows us to move them to GPU asynchronously (non_blocking=True)
    x, y = (
        x.pin_memory().to(device, non_blocking=True),
        y.pin_memory().to(device, non_blocking=True),
    )
    return x, y

In [71]:
torch.random.manual_seed(1024)
x, y = get_batch(data_dir)

In [72]:
model(x)

(tensor([[[ 0.0948,  0.0907, -0.1434,  ..., -0.0196,  0.0444,  0.0897]]],
        device='cuda:7', grad_fn=<UnsafeViewBackward0>),
 None)